# FINTECH545 Practice Midterm 1 — Claude's solution

Every calculation uses Roger's `Library/` package (imported via `Risk_Library`) wherever it has a matching function.
Where the Library has no matching function, the small extra step is written out inline and marked **(inline: not in Library)**.

At the end of each problem, a **check cell** compares the result with the reference answers in `answers.jl`.

In [2]:
import numpy as np
import pandas as pd
from scipy import stats

# Everything comes from the aggregated Library entry point
from Risk_Library import (
    first_four_moments,
    fit_normal_distribution,
    fit_t_distribution,
    t_distribution_aicc,
    calculate_var,
    calculate_es,
    ew_covariance_matrix,
    covariance_matrix,
    classify_psd,
    fix_psd_matrix,
    return_calculate,
    simulate_copula,
    calculate_portfolio_var_es,
)
from utils import aicc   # shared AICc helper (Library/utils.py)

pd.set_option("display.precision", 6)

## Problem 1 (10 pts)
*Explain the difference in thinking between data modeling for risk analysis vs data modeling for forecasting.*

**Forecasting** cares about the *center* of the future distribution: the goal is a point prediction (the conditional
expected value), and a model is judged by how close that prediction is to what happens (e.g. MSE). Noise around
the prediction is mostly treated as a nuisance.

**Risk analysis** cares about the *whole* distribution of future outcomes, especially the **tails**. The mean is
often nearly irrelevant (daily returns have mean ≈ 0). What matters is the spread (variance), asymmetry (skewness),
tail thickness (kurtosis), and how assets move together in extreme states (dependence/copulas). A risk model is judged on
whether it gives the right *probabilities* of bad outcomes (VaR/ES, backtesting exceedances), not on whether it
predicts tomorrow's value. So the risk modeler chooses distributions (normal vs t, etc.) and dependence
structures that capture these higher moments and the joint tail behavior.

## Problem 2 (20 pts) — `problem2.csv`
a. Mean, Variance, Skewness, Kurtosis (8)
b. Normal or t, based on (a) alone? (4)
c. Fit both and prove/disprove (b) (8)

In [3]:
data2 = pd.read_csv("problem2.csv")["X"]

# (a) First four moments with the Library.
#  - method="biased": population skewness/kurtosis formulas (same as Julia's StatsBase,
#    which is what answers.jl uses). method="unbiased" (pandas/Excel style) shown too for reference.
#  - kurtosis_type="excess": normal distribution -> 0, so a value > 0 means fat tails.
moments_biased = first_four_moments(data2, method="biased", kurtosis_type="excess")
moments_unbiased = first_four_moments(data2, method="unbiased", kurtosis_type="excess")
pd.DataFrame({"biased (answer)": moments_biased, "unbiased (reference)": moments_unbiased})

,biased (answer),unbiased (reference)
Mean,-0.000346,-0.000346
Variance,0.000485,0.000485
Skewness,0.114081,0.114253
Kurtosis,0.957560,0.968389


Note: `method="biased"` uses the population variance (divide by n). The textbook sample variance (n−1) is
the "unbiased" column; at this sample size (n = 1000) the two agree to 6 significant digits. The answer key reports
the sample variance with the biased skewness/kurtosis.

**(b)** Skewness is small (≈ 0.11, close to symmetric), but the **excess kurtosis ≈ 0.96 > 0**. That means
fatter tails than a normal distribution. Based on (a) alone, I would choose the **t-distribution**, since a normal distribution has
excess kurtosis 0 and cannot produce fat tails, while the t distribution's degrees of freedom ν control tail thickness.

In [3]:
# (c) Fit both distributions with the Library (MLE for t; mean/sample-std for the normal)
normal_params = fit_normal_distribution(data2)      # mu, sigma
t_params = fit_t_distribution(data2)                # mu, sigma, nu
display(pd.DataFrame({"Normal": normal_params, "T": t_params}))

# Compare the fits with AICc (lower = better, penalised for the number of parameters).
# - t: Library computes it directly (3 params: mu, sigma, nu).
# - normal (inline: not in Library as a one-call function): log-likelihood under the fitted
#   normal, then the shared Library helper utils.aicc with 2 params (mu, sigma).
normal_loglik = stats.norm.logpdf(data2, loc=normal_params["mu"], scale=normal_params["sigma"]).sum()
normal_aicc = aicc(normal_loglik, num_params=2, num_obs=len(data2))
t_aicc = t_distribution_aicc(data2, fitted=t_params)

aicc_table = pd.Series({"Normal AICc": normal_aicc, "T AICc": t_aicc})
display(aicc_table)
print("Better fit:", "T-distribution" if t_aicc < normal_aicc else "Normal distribution")

,Normal,T
mu,-0.000346,-0.000478
nu,NaN,8.859360
sigma,0.022032,0.019399


Normal AICc   -4789.600319
T AICc        -4809.558174
dtype: float64

Better fit: T-distribution


**(c)** The t-distribution has the lower AICc (≈ −4809.6 vs ≈ −4789.6, a gap of ≈ 20, which is large). It is clearly the better
fit even after paying the penalty for the extra parameter ν. The fitted ν ≈ 8.9 confirms moderately fat tails.
This **confirms** the choice in (b).

In [4]:
# Check against answers.jl
ref2 = {"Mean": -0.0003457749550813159, "Variance": 0.00048542322361448215,
        "Skewness": 0.11408120646692742, "Kurtosis": 0.9575595460250139,
        "Normal AICc": -4789.600319016606, "T AICc": -4809.558173851321}
mine2 = {"Mean": moments_biased["Mean"], "Variance": moments_unbiased["Variance"],
         "Skewness": moments_biased["Skewness"], "Kurtosis": moments_biased["Kurtosis"],
         "Normal AICc": normal_aicc, "T AICc": t_aicc}
pd.DataFrame({"mine": mine2, "answers.jl": ref2}).assign(match=lambda d: np.isclose(d["mine"], d["answers.jl"], rtol=1e-6))

,mine,answers.jl,match
Mean,-0.000346,-0.000346,True
Variance,0.000485,0.000485,True
Skewness,0.114081,0.114081,True
Kurtosis,0.957560,0.957560,True
Normal AICc,-4789.600319,-4789.600319,True
T AICc,-4809.558174,-4809.558174,True


## Problem 3 (20 pts) — `problem2.csv` returns, fitted models
a. VaR (α = 5%) as distance from 0, both models (8)
b. ES (α = 5%) as distance from 0, both models (8)
c. Discuss (4)

In [5]:
# "Distance from 0" = the Library's "Absolute" output: VaR = -quantile_alpha(R), ES = -E[R | R <= VaR quantile].
# (The "Diff from Mean" output would measure from the fitted mean instead.)
# method="analytical": closed-form quantile / tail expectation of the fitted distribution, no simulation noise.
risk3 = pd.DataFrame({
    dist: {
        "VaR (5%)": calculate_var(data2, distribution=dist, method="analytical", alpha=0.05)["VaR Absolute"],
        "ES (5%)":  calculate_es(data2, distribution=dist, method="analytical", alpha=0.05)["ES Absolute"],
    }
    for dist in ["normal", "t"]
})
risk3

,normal,t
VaR (5%),0.036586,0.036102
ES (5%),0.045792,0.048230


**(c) Discussion.**
- **VaR is almost the same** for both models (≈ 3.66% normal vs ≈ 3.61% t). The t VaR is even slightly *smaller*.
- **ES is noticeably larger under the t** (≈ 4.82% vs ≈ 4.58%).

Why: the two models are fit to the same data, so they must agree on the overall spread. The t-distribution reaches
that variance with a narrower center and **fatter tails** (more extreme events). At the 5% quantile, which is not very far
into the tail, the two distributions cross at nearly the same point, so VaR barely changes (the t even puts slightly more
mass near the center). ES averages over *everything beyond* that quantile, so the extra extreme-loss mass of the t pulls
the average loss out. This is why VaR alone can understate tail risk, and why ES (which is coherent and tail-sensitive) shows
the difference between a thin-tailed and a fat-tailed model.

In [6]:
# Check against answers.jl
ref3 = pd.DataFrame({"normal": {"VaR (5%)": 0.036585720113925815, "ES (5%)": 0.04579212823081352},
                     "t":      {"VaR (5%)": 0.03610245171424064,  "ES (5%)": 0.04823020079076017}})
print(risk3.round(6).equals(ref3.round(6)) or np.allclose(risk3, ref3, rtol=1e-5))
ref3

True


,normal,t
VaR (5%),0.036586,0.036102
ES (5%),0.045792,0.048230


## Problem 4 (10 pts) — `problem4.csv`
a. EW correlation, λ = 0.94 (3)
b. EW variances, λ = 0.97 (3)
c. Combine into a covariance matrix (3)
d. Why? (1)

In [7]:
data4 = pd.read_csv("problem4.csv")   # rows ordered oldest -> newest (what the Library expects)

# (a) Exponentially weighted correlation matrix with lambda = 0.94
ew_corr_094 = ew_covariance_matrix(data4, lambda_=0.94, method="correlation")
ew_corr_094

,x1,x2,x3
x1,1.000000,0.711329,0.807175
x2,0.711329,1.000000,0.713020
x3,0.807175,0.713020,1.000000


In [8]:
# (b) Exponentially weighted variances with lambda = 0.97 = the diagonal of the lambda=0.97 EW covariance
ew_var_097 = pd.Series(np.diag(ew_covariance_matrix(data4, lambda_=0.97, method="covariance")),
                       index=data4.columns, name="EW variance (λ=0.97)")
ew_var_097

x1    0.015379
x2    0.035517
x3    0.027813
Name: EW variance (λ=0.97), dtype: float64

In [9]:
# (c) Combine: Cov = diag(sigma) @ Corr @ diag(sigma), with sigma from the lambda=0.97 variances and
# Corr from lambda=0.94. The Library does exactly this mix in one call:
#   lambda_ = decay used for the variances, lambda_corr = decay used for the correlation.
mixed_cov = ew_covariance_matrix(data4, lambda_=0.97, method="covariance", lambda_corr=0.94)

# Sanity check: same matrix built by hand from the (a) and (b) pieces
sd = np.sqrt(ew_var_097.values)
assert np.allclose(mixed_cov.values, np.outer(sd, sd) * ew_corr_094.values)
mixed_cov

,x1,x2,x3
x1,0.015379,0.016625,0.016694
x2,0.016625,0.035517,0.022410
x3,0.016694,0.022410,0.027813


**(d)** In practice, volatility and correlation don't update at the same speed, so they may deserve different
memories. A lower λ (0.94) gives more weight to recent observations, so the correlation estimate reacts quickly to changes
in how assets co-move (e.g. correlations jumping in a crisis). A higher λ (0.97) keeps more history, so the
variance estimate is more stable and less noisy. Mixing them gives a risk manager control over each piece separately
(RiskMetrics-style practice), while the result is still a valid PSD covariance matrix, because a scaled valid correlation matrix stays PSD.

In [10]:
# Check against answers.jl
ref4_corr = np.array([[1.0, 0.711329, 0.807175], [0.711329, 1.0, 0.71302], [0.807175, 0.71302, 1.0]])
ref4_var = np.array([0.015378811498568219, 0.03551743200920777, 0.02781346161629677])
ref4_cov = np.array([[0.0153788, 0.0166247, 0.0166938], [0.0166247, 0.0355174, 0.0224104], [0.0166938, 0.0224104, 0.0278135]])
print("corr match:", np.allclose(ew_corr_094, ref4_corr, atol=1e-6))
print("var  match:", np.allclose(ew_var_097, ref4_var, rtol=1e-9))
print("cov  match:", np.allclose(mixed_cov, ref4_cov, atol=1e-7))

corr match: True
var  match: True
cov  match: True


## Problem 5 (30 pts) — `problem5.csv` (has missing values)
a. Pairwise covariance (10)
b. PD / PSD / non-definite? (5)
c. If non-definite, fix with Higham (10)
d. Variance explained & cumulative per principal component, largest → smallest (5)

In [5]:
data5 = pd.read_csv("problem5.csv")
print("Missing values per column:")
print(data5.isna().sum().to_string())

# (a) Pairwise covariance: each (i, j) entry uses all rows where BOTH column i and j are present.
pairwise_cov = covariance_matrix(data5, method="covariance", missing_strategy="pairwise")
pairwise_cov

Missing values per column:
x1    13
x2    11
x3    15
x4    15
x5    13


,x1,x2,x3,x4,x5
x1,1.470484,1.454214,0.877269,1.903226,1.444361
x2,1.454214,1.252078,0.539548,1.621918,1.237877
x3,0.877269,0.539548,1.272425,1.171959,1.091912
x4,1.903226,1.621918,1.171959,1.814469,1.589729
x5,1.444361,1.237877,1.091912,1.589729,1.396186


In [6]:
# (b) Classify the matrix with the Library diagnostic (eigenvalue-based, scale-aware tolerance)
psd_report = classify_psd(pairwise_cov)
print(psd_report["classification"])
print(psd_report["summary"])
print("Eigenvalues of the covariance matrix:", np.round(psd_report["eigenvalues"], 6))

# Same verdict on the matching correlation matrix (inline: rescaling only), since rescaling by
# positive standard deviations cannot change the sign of the eigenvalues (Sylvester's law of inertia)
sd5 = np.sqrt(np.diag(pairwise_cov))
pairwise_corr_from_cov = pairwise_cov / np.outer(sd5, sd5)
print("Min eigenvalue of cov2cor(C):", np.linalg.eigvalsh(pairwise_corr_from_cov.values).min())

non-PSD
non-PSD: 2 eigenvalue(s) below -threshold (6.79e-08): [-0.3102, -0.1332].
Eigenvalues of the covariance matrix: [-0.310243 -0.133232  0.027978  0.834434  6.786706]
Min eigenvalue of cov2cor(C): -0.19179203172600037


**(b)** The pairwise covariance matrix has **negative eigenvalues** (two of them: ≈ −0.31 and ≈ −0.13), so it is
**non-definite** (neither PD nor PSD). This is not a valid covariance matrix: some portfolio would get a
negative variance. This happens because each entry was estimated on a *different* subset of rows.

*(The answer key prints −0.0948 as the smallest eigenvalue of cov2cor(C). In answers.jl that line runs `missing_cov(x; ...)` on a
variable `x` instead of the loaded `data`, so it looks like a typo in the key. The conclusion, non-definite, is the same either way.)*

In [7]:
# (c) Higham's alternating-projections nearest-PSD fix. For a covariance input the Library converts it to a
# correlation matrix, finds the nearest PSD correlation matrix, then rescales back with the original
# variances, so the diagonal (variances) is unchanged.
higham_cov = fix_psd_matrix(pairwise_cov, method="higham")
display(higham_cov)
print(classify_psd(higham_cov)["summary"])

,x1,x2,x3,x4,x5
x1,1.470484,1.332361,0.884378,1.627602,1.399556
x2,1.332361,1.252078,0.619028,1.450604,1.214450
x3,0.884378,0.619028,1.272425,1.076846,1.059658
x4,1.627602,1.450604,1.076846,1.814469,1.577928
x5,1.399556,1.214450,1.059658,1.577928,1.396186


PSD: all eigenvalues are above -threshold (6.47e-08); 3 eigenvalue(s) within threshold of zero; 3 small negative eigenvalue(s) were treated as zero.


In [8]:
# (d) PCA on the fixed covariance matrix (inline: no PCA-explained-variance function in Library).
# Eigenvalues of a covariance matrix = variance along each principal component.
eigenvalues = np.sort(np.linalg.eigvalsh(higham_cov.values))[::-1]     # largest -> smallest
explained = eigenvalues / eigenvalues.sum()
print(explained)
pca_table = pd.DataFrame({
    "PC": np.arange(1, len(eigenvalues) + 1),
    "Eigenvalue": eigenvalues,
    "Explained": explained,
    "Cumulative": np.cumsum(explained),
}).set_index("PC")
pca_table

[ 8.97765734e-01  1.02234267e-01 -1.19044346e-10 -1.86448134e-10
 -2.56340907e-10]


,Eigenvalue,Explained,Cumulative
PC,,,
1,6.468979e+00,8.977657e-01,0.897766
2,7.366636e-01,1.022343e-01,1.000000
3,-8.577911e-10,-1.190443e-10,1.000000
4,-1.343479e-09,-1.864481e-10,1.000000
5,-1.847101e-09,-2.563409e-10,1.000000


**(d)** PC1 explains ≈ 89.8% and PC2 ≈ 10.2% (cumulative 100%). PCs 3–5 explain ≈ 0. Their eigenvalues are
~1e-10, which is numerical zero: Higham projected the negative eigenvalues up to 0. So the repaired matrix is (numerically) rank 2, PSD but not PD.

In [15]:
# Check against answers.jl
ref5_cov = np.array([[1.47048, 1.45421, 0.877269, 1.90323, 1.44436],
                     [1.45421, 1.25208, 0.539548, 1.62192, 1.23788],
                     [0.877269, 0.539548, 1.27242, 1.17196, 1.09191],
                     [1.90323, 1.62192, 1.17196, 1.81447, 1.58973],
                     [1.44436, 1.23788, 1.09191, 1.58973, 1.39619]])
ref5_higham = np.array([[1.47048, 1.33236, 0.884378, 1.6276, 1.39956],
                        [1.33236, 1.25208, 0.619028, 1.4506, 1.21445],
                        [0.884378, 0.619028, 1.27242, 1.07685, 1.05966],
                        [1.6276, 1.4506, 1.07685, 1.81447, 1.57793],
                        [1.39956, 1.21445, 1.05966, 1.57793, 1.39619]])
ref5_explained = np.array([0.897766, 0.102234, 0, 0, 0])
print("pairwise cov match:", np.allclose(pairwise_cov, ref5_cov, atol=1e-5))
print("higham cov match:  ", np.allclose(higham_cov, ref5_higham, atol=1e-5))
print("explained match:   ", np.allclose(explained, ref5_explained, atol=1e-6))

pairwise cov match: True
higham cov match:   True
explained match:    True


## Problem 6 (20 pts) — `problem6.csv` prices, 100 shares of each stock, arithmetic returns
a. De-mean returns, fit a Student t per stock, report fits (3)
b. Simulate with a Gaussian copula and report the correlation matrix used (3)
c. VaR & ES (5%) per stock in $ (7)
d. VaR & ES (5%) for the total portfolio in $ (7)

In [16]:
prices6 = pd.read_csv("problem6.csv")
stocks = ["x1", "x2", "x3"]

# Arithmetic returns: R_t = P_t / P_{t-1} - 1  (Date column carried along, then dropped)
returns6 = return_calculate(prices6, method="arithmetic", date_column="Date").set_index("Date")[stocks]

# (a) De-mean each series so its sample mean is exactly 0
returns6 = returns6 - returns6.mean()
print("Means after de-meaning:", returns6.mean().round(15).to_dict())

# Fit a Student t (MLE) to each stock's de-meaned returns
t_fits6 = pd.DataFrame({s: fit_t_distribution(returns6[s]) for s in stocks}).T
t_fits6

Means after de-meaning: {'x1': 0.0, 'x2': 0.0, 'x3': -0.0}


,mu,sigma,nu
x1,-0.000479,0.012908,4.729831
x2,-0.000044,0.009058,6.766945
x3,0.000075,0.017063,39.864293


**(a)** Fitted t parameters per stock are shown above (μ, σ, ν). The MLE location μ is not exactly 0 even though the sample mean is 0,
because the t location is a robust "center", not the sample mean. x1 has the fattest tails (ν ≈ 4.7), x2 is moderate (ν ≈ 6.8), and
x3 is nearly normal (ν ≈ 40).

In [17]:
# (b) Gaussian copula simulation with the Library:
#   1. each column gets its fitted t marginal (marginals="t", same MLE fit as part a),
#   2. historical returns -> uniforms U = F_t(r) via each stock's own fitted CDF,
#   3. copula correlation = Spearman rank correlation (correlation="spearman"),
#   4. draw correlated normals Z ~ N(0, corr), U = Phi(Z), returns = F_t^-1(U) per stock.
NUM_SIMS = 1_000_000
sim_returns6, copula_fit6 = simulate_copula(
    returns6, copula="gaussian", marginals="t", correlation="spearman",
    num_simulations=NUM_SIMS, seed=545, output="both",
)
copula_corr6 = copula_fit6["correlation"]
print("Correlation matrix used in the Gaussian copula (Spearman):")
copula_corr6

Correlation matrix used in the Gaussian copula (Spearman):


,x1,x2,x3
x1,1.000000,0.446299,0.394197
x2,0.446299,1.000000,0.511761
x3,0.394197,0.511761,1.000000


**(b)** The copula correlation is the Spearman rank correlation shown above. Ranks are unchanged by monotone transforms, so this is the
same as the Spearman correlation of the fitted uniforms U = F(r), which is exactly what the answer key uses.

In [18]:
# (c) + (d) Turn the simulated returns into $ P&L (inline: Library's simulate_copula returns returns, not P&L)
current_prices6 = prices6[stocks].iloc[-1]          # most recent price of each stock
holdings6 = pd.Series(100, index=stocks)            # 100 shares of each
current_value6 = holdings6 * current_prices6        # $ value of each position today

# Simulated P&L of each position = value * simulated arithmetic return; total = sum across stocks
pnl6 = sim_returns6 * current_value6
pnl6["Total"] = pnl6.sum(axis=1)
value6 = pd.concat([current_value6, pd.Series({"Total": current_value6.sum()})])

def empirical_var_es(pnl, alpha=0.05):
    # VaR = -(alpha quantile of P&L); ES = -(average P&L at or below that quantile)
    cutoff = np.quantile(pnl, alpha)
    return -cutoff, -pnl[pnl <= cutoff].mean()

risk6 = pd.DataFrame(
    {col: empirical_var_es(pnl6[col].values) for col in pnl6.columns}, index=["VaR95 ($)", "ES95 ($)"]
).T
risk6["Current Value ($)"] = value6
risk6["VaR95 (%)"] = risk6["VaR95 ($)"] / risk6["Current Value ($)"]
risk6["ES95 (%)"] = risk6["ES95 ($)"] / risk6["Current Value ($)"]
risk6

,VaR95 ($),ES95 ($),Current Value ($),VaR95 (%),ES95 (%)
x1,222.544175,321.536206,8306.321359,0.026792,0.038710
x2,133.609099,183.339739,7740.486139,0.017261,0.023686
x3,235.298330,298.998653,8215.614780,0.028640,0.036394
Total,475.472663,631.960076,24262.422278,0.019597,0.026047


In [19]:
# Cross-check with the Library's one-call portfolio function (Gaussian copula with t marginals).
# Differences from the table above: it builds the copula correlation from normal scores (Pearson of Phi^-1(U))
# rather than Spearman, and it uses its own random draws, so the numbers differ slightly but should agree closely.
portfolio6 = pd.DataFrame({"Stock": stocks, "Holding": 100,
                           "Starting Price": current_prices6.values, "Distribution": "T"})
calculate_portfolio_var_es(portfolio6, returns6, alpha=0.05, num_simulations=NUM_SIMS, seed=545)

,Stock,VaR95,ES95,VaR95_Pct,ES95_Pct
0,x1,222.544175,321.536206,0.026792,0.038710
1,x2,133.563939,183.352712,0.017255,0.023687
2,x3,235.337271,299.018711,0.028645,0.036396
3,Total,481.589538,641.025248,0.019849,0.026420


**(c)** Per-stock $ VaR / ES (5%) are in the `x1`–`x3` rows of the table above (≈ \$222 / \$321 for x1, ≈ \$134 / \$184 for x2,
≈ \$235 / \$299 for x3).

**(d)** Total-portfolio VaR ≈ \$475 and ES ≈ \$632. Both are well below the sum of the individual numbers (VaR sum ≈ \$591,
ES sum ≈ \$804). This is the diversification benefit, because the stocks are only moderately correlated (ρ ≈ 0.4–0.5).

In [20]:
# Check against answers.jl: these are Monte-Carlo results, so instead of an exact match they should land inside the
# answer key's 95% confidence bands (built from 1000 runs of 1000 sims). Point estimates from 1M sims are listed for reference.
ref6 = pd.DataFrame({
    "VaR_CL025": [202.028, 121.519, 217.198, 434.27], "VaR_CL975": [246.906, 148.029, 256.725, 521.733],
    "ES_CL025":  [283.99, 163.674, 276.384, 574.285], "ES_CL975":  [364.389, 203.744, 321.952, 686.507],
    "Key VaR95": [222.714, 133.956, 235.331, 477.606], "Key ES95":  [321.864, 183.998, 298.907, 632.105],
}, index=["x1", "x2", "x3", "Total"])
ref6["My VaR95"] = risk6["VaR95 ($)"]
ref6["My ES95"] = risk6["ES95 ($)"]
ref6["VaR in CI"] = ref6["My VaR95"].between(ref6["VaR_CL025"], ref6["VaR_CL975"])
ref6["ES in CI"] = ref6["My ES95"].between(ref6["ES_CL025"], ref6["ES_CL975"])
ref6

,VaR_CL025,VaR_CL975,ES_CL025,ES_CL975,Key VaR95,Key ES95,My VaR95,My ES95,VaR in CI,ES in CI
x1,202.028,246.906,283.990,364.389,222.714,321.864,222.544175,321.536206,True,True
x2,121.519,148.029,163.674,203.744,133.956,183.998,133.609099,183.339739,True,True
x3,217.198,256.725,276.384,321.952,235.331,298.907,235.298330,298.998653,True,True
Total,434.270,521.733,574.285,686.507,477.606,632.105,475.472663,631.960076,True,True
